The libraries are considering the native databricks enviroment at 18-03-2026

In [ ]:
#%pip install xgboost
from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup
import mlflow
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
import pandas as pd

###  Create the training set via FeatureLookup
The FeatureLookup pattern is the core concept that interviewers test. Instead of manually joining feature tables to your labels, the Feature Engineering client creates a TrainingSet that records which features came from which table and at what key. This linkage is embedded in the logged MLflow model — at inference time, the model retrieves features automatically.

In [ ]:
fe = FeatureEngineeringClient()


# Load labels
labels_df = spark.table('ml1.ml_project.churn_labels')


# Define feature lookups — this is the key MLE pattern
feature_lookups = [
    FeatureLookup(
        table_name='ml1.ml_project.customer_rfm_features',
        lookup_key='customer_id',
        feature_names=['recency_days', 'frequency', 'monetary', 'avg_order_value']
    )
]


# Create training set — joins labels with features from the store
training_set = fe.create_training_set(
    df=labels_df,
    feature_lookups=feature_lookups,
    label='purchased_next_90d',
    exclude_columns=['customer_id']
)


training_df = training_set.load_df().toPandas()
print(f'Training set shape: {training_df.shape}')
print(training_df.head())


## Train and track with MLflow 3
MLflow 3 is pre-installed on Databricks Free Edition.

In [ ]:
from mlflow.models.signature import infer_signature


# Features and target
feature_cols = ['recency_days', 'frequency', 'monetary', 'avg_order_value']
X = training_df[feature_cols]
y = training_df['purchased_next_90d']


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


# Set experiment — creates it in Databricks MLflow UI if it doesn't exist
mlflow.set_experiment('/Users/brandon5thb@gmail.com/churn_prediction_rfm')


with mlflow.start_run(run_name='xgb_baseline') as run:


    # Hyperparameters
    params = {
        'n_estimators': 200,
        'max_depth': 4,
        'learning_rate': 0.05,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'scale_pos_weight': (y_train == 0).sum() / (y_train == 1).sum(),  # handle class imbalance
        'random_state': 42,
        'eval_metric': 'auc',
        'use_label_encoder': False
    }
    mlflow.log_params(params)


    # Train
    model = xgb.XGBClassifier(**params)
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)


    # Evaluate
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_pred_proba)
    mlflow.log_metric('roc_auc', auc)
    print(f'ROC-AUC: {auc:.4f}')


    # Log feature importance
    importance = dict(zip(feature_cols, model.feature_importances_))
    mlflow.log_dict(importance, 'feature_importance.json')


    # Log model WITH feature store linkage
    # This is critical — it embeds the FeatureLookup metadata in the model artifact
    signature = infer_signature(X_train, model.predict(X_train))


    fe.log_model(
        model=model,
        artifact_path='churn_model',
        flavor=mlflow.xgboost,
        training_set=training_set,  # <-- this links features to the model
        registered_model_name='ml1.ml_project.churn_prediction_model',
        signature=signature
    )


    print(f'Run ID: {run.info.run_id}')
    print(f'Model registered in Unity Catalog as: ml1.ml_project.churn_prediction_model')